# Class-conditional image generation and editing using MaskGIT.

This notebook is an official Colab notebook using pretrained [MaskGIT](https://arxiv.org/pdf/2202.04200.pdf) models for class-conditional image generation.

After connecting to a runtime, get started by following these instructions:

1. Make sure you've selected a GPU accelerator (Runtime > Change runtime type > Hardware accelerator > GPU).
2.Click the **Play** button to the left of the code cell, or use the keyboard shortcut "Command/Ctrl+Enter" to generate.

Install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install jax flax
%pip install numpy tensorflow matplotlib ml_collections
%pip install scikit-commpy

Clone the repository


In [ ]:
# 刪除舊資料夾
!rm -rf /content/maskgit
# 回到根目錄以防萬一
%cd /content/
!git clone -b feat/jax-compat https://github.com/zhengpohung/maskgit
%cd maskgit
%ls

Set up a couple of imports before we dive in.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
if not hasattr(jnp, "DeviceArray"):
    jnp.DeviceArray = jax.Array
import os
import itertools
from timeit import default_timer as timer

import maskgit
from maskgit.utils import visualize_images, read_image_from_url, restore_from_path, draw_image_with_bbox, Bbox
from maskgit.inference import ImageNet_class_conditional_generator
from communication_sim_ARQ import simulate_transmission_analytical
import tensorflow_datasets as tfds
import tensorflow as tf

Download the pretrained models, including

1. two tokenization models, both of which map an image of size $H \times W$ into a latent code of size $H/16 \times W/16$.
2. two masked visual token modeling (MVTM) models, one for $256 \times 256$ and one for $512 \times 512$.


In [ ]:
!mkdir -p checkpoints/

models_to_download = itertools.product(
    *[ ["maskgit", "tokenizer"],   [256, 512] ])

for (type_, resolution) in models_to_download:
  canonical_path = ImageNet_class_conditional_generator.checkpoint_canonical_path(type_, resolution)
  if os.path.isfile(canonical_path):
    print(f"Checkpoint for {resolution} {type_} already exists, not downloading again")
  else:
    source_url = f'https://storage.googleapis.com/maskgit-public/checkpoints/{type_}_imagenet{resolution}_checkpoint'
    !wget {source_url} -O {canonical_path}

### Initialization & resoultion defaults
Instantiate maskgit generators for both resolutions using the pre-trained checkpoints we just downloaded. By default, the examples for generation use the 256 model, while the image editing examples use the 512 model for demo purposes.

### Run Mode

Let's also choose a **run_mode**, which can be either 'normal' or 'pmap'. By default, 'normal' is enabled.

The 'pmap' mode uses [jax.pmap] under the hood, which offers a substantial  speedup for accelerators that can handle the higher memory
requirement, e.g. TPUs, or GPUs such as V100.

Running 'pmap' mode with a GPU (or CPU) with a smaller memory may lead to OOM crashes. That said,

**if your hardware allows, 'pmap' mode is strongly recommended. Each image can typically be generated in < 1s in 'pmap' mode on a TPU**.

In [ ]:
generator_256 = ImageNet_class_conditional_generator(image_size=256)
generator_512 = ImageNet_class_conditional_generator(image_size=512)
arbitrary_seed = 42
rng = jax.random.PRNGKey(arbitrary_seed)

run_mode = 'normal'  #@param ['normal', 'pmap']

p_generate_256_samples = generator_256.p_generate_samples()
p_edit_512_samples = generator_512.p_edit_samples()

# Class-conditional Image Synthesis

Choose the ImageNet **label**, which determines what type of object to synthesize.

In [ ]:
import os

# --- 路徑設定 ---
# 根目錄指向您在雲端硬碟中的資料夾
drive_root = '/content/drive/MyDrive/MaskGIT'

# JSON 檔案與圖片資料夾的完整路徑
json_path = os.path.join(drive_root, 'class_to_folder.json')
image_folder_root = os.path.join(drive_root, 'ImageNet100')

# 檢查路徑是否存在
if os.path.exists(drive_root):
    print(f"成功連接到雲端硬碟資料夾: {drive_root}")
    if not os.path.exists(json_path):
        print(f"⚠️ 警告: 找不到 JSON 檔案於 {json_path}")
    if not os.path.exists(image_folder_root):
        print(f"⚠️ 警告: 找不到 ImageNet100 資料夾於 {image_folder_root}")
else:
    print(f"❌ 錯誤: 找不到您的資料夾 {drive_root}，請確認路徑是否正確。")


In [ ]:
import json
from PIL import Image
import os
import random
import math
import gc

# --- 路徑設定 ---
# 【修改點】: 將儲存根目錄指回您的 Google 雲端硬碟
drive_root = '/content/drive/MyDrive/MaskGIT'
results_save_root = os.path.join(drive_root, 'simulation_results')
os.makedirs(results_save_root, exist_ok=True)
print(f"所有重建圖片將儲存於: {results_save_root}")

# --- 實驗參數 ---
image_size = 256
snr_values = [6, 8, 10, 12, 14, 16, 18]
model_mask_id = generator_256.maskgit_cf.transformer.mask_token_id
#model_mask_id = 0

# --- 讀取類別對應檔並抽樣 ---
try:
    with open(json_path, 'r', encoding='utf-8') as f:
        class_mapping = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"錯誤：找不到 JSON 檔案於 '{json_path}'。請確認路徑是否正確。")

all_image_paths = []
for label_str, folder_name in class_mapping.items():
    class_folder_path = os.path.join(image_folder_root, folder_name)
    if os.path.isdir(class_folder_path):
        for image_name in os.listdir(class_folder_path):
            if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_image_paths.append((os.path.join(class_folder_path, image_name), int(label_str), image_name))

print(f"共找到 {len(all_image_paths)} 張圖片。")
random.seed(156)
random.shuffle(all_image_paths)
num_to_process = math.ceil(len(all_image_paths) * 1)
image_subset = all_image_paths[:num_to_process]
print(f"共 {len(image_subset)} 張圖片。")

# --- 主迴圈：高效批次處理 + 可續跑功能 ---
rng, sample_rng = jax.random.split(rng)
start_timer = timer()
total_image_count = 0
skipped_count = 0

def preprocess_image_direct_random_crop(image_path, target_size=256):
    """
    執行 Direct Random Crop 預處理 (不縮放)：
    1. 直接從原始圖片中隨機裁切 target_size x target_size 區域。
    2. 如果圖片小於 target_size，將引發錯誤。
    """
    img = Image.open(image_path).convert('RGB')
    width, height = img.size

    # 1. 檢查圖片是否足夠大
    if width < target_size or height < target_size:
        raise ValueError(f"Image is too small ({width}x{height}) for direct {target_size}x{target_size} crop.")

    # 2. 計算隨機裁切座標 (邏輯同您的 cv2 參考)
    max_left = width - target_size
    max_top = height - target_size

    # 隨機選擇左上角座標
    left = random.randint(0, max_left)
    top = random.randint(0, max_top)
    right = left + target_size
    bottom = top + target_size

    # 執行隨機裁切
    img_cropped = img.crop((left, top, right, bottom))

    return img_cropped

for image_path, source_label, original_filename in image_subset:
    # --- !!【新增的可續跑邏輯】!! ---
    # 檢查這張圖的其中一個最終檔案是否存在於雲端硬碟
    # 我們選擇檢查 SNR=18 的 CMI 版本圖片
    last_snr = snr_values[-1]
    check_filename = f"{source_label}_{os.path.splitext(original_filename)[0]}.png"
    check_path = os.path.join(results_save_root, f"SNR_{last_snr}", "with_cmi", check_filename)

    if os.path.exists(check_path):
        if total_image_count % 1 == 0: # 每隔一段時間提示一次，避免洗版
            print(f"  - 第 {total_image_count + 1} 張圖片 '{original_filename}' 已處理過，跳過。")
        skipped_count += 1
        total_image_count += 1
        continue
    # -----------------------------------------------

    print(f"  - 正在處理第 {total_image_count + 1} / {len(image_subset)} 張圖片: {original_filename}")





    try:
        img = preprocess_image_direct_random_crop(image_path, image_size)
        source_image_np = np.array(img)
        source_image_ex = np.expand_dims(source_image_np.astype(np.float32) / 255.0, axis=0)
        #print("image before encoding = ", source_image_ex.shape, source_image_ex)
    except Exception as e:
        print(f"讀取圖片 {image_path} 失敗: {e}")
        total_image_count += 1
        continue

    # 1. 編碼一次，得到 perfect_tokens
    _, result_dict = generator_256.tokenizer_model.apply(
        {'params': generator_256.tokenizer_variables['params']},
        {"image": source_image_ex},
        method=generator_256.tokenizer_model.encode
    )
    perfect_tokens = result_dict['encoding_indices']
    #print("image after encoding = ", perfect_tokens.shape, perfect_tokens)
    # 2. 準備批次
    batch_input_wc, batch_input_woc = [], []
    codebook_size = generator_256.maskgit_cf.vqvae.codebook_size
    for snr_db in snr_values:
        tokens_after_channel = simulate_transmission_analytical(perfect_tokens, snr_db, generator_256)
        #print("tokens_after_channel = ", tokens_after_channel.shape, tokens_after_channel)
        label_token_wc = np.array([[source_label + codebook_size]])
        batch_input_wc.append(np.concatenate([label_token_wc, tokens_after_channel], axis=1))
        label_token_woc = np.array([[model_mask_id]])
        batch_input_woc.append(np.concatenate([label_token_woc, tokens_after_channel], axis=1))
    final_batch_wc = np.concatenate(batch_input_wc, axis=0)
    final_batch_woc = np.concatenate(batch_input_woc, axis=0)

    # 3. 模型推理
    results_with_cmi = generator_256.generate_samples(input_tokens=final_batch_wc, rng=sample_rng)
    results_without_cmi = generator_256.generate_samples(input_tokens=final_batch_woc, rng=sample_rng)
    #print("results_with_cmi = ", results_with_cmi.shape, results_with_cmi)
    #print("results_without_cmi = ", results_without_cmi.shape, results_without_cmi)
    # 4. 儲存結果到【雲端硬碟】
    for i, snr_db in enumerate(snr_values):
        snr_folder = os.path.join(results_save_root, f"SNR_{snr_db}")
        os.makedirs(os.path.join(snr_folder, "original"), exist_ok=True)
        os.makedirs(os.path.join(snr_folder, "with_cmi"), exist_ok=True)
        os.makedirs(os.path.join(snr_folder, "without_cmi"), exist_ok=True)
        base_filename = f"{source_label}_{os.path.splitext(original_filename)[0]}.png"
        original_save_path = os.path.join(snr_folder, "original", base_filename)
        if not os.path.exists(original_save_path):
            Image.fromarray(source_image_np).save(original_save_path)
        reconstructed_wc = (np.clip(results_with_cmi[i], 0, 1) * 255).astype(np.uint8)
        reconstructed_woc = (np.clip(results_without_cmi[i], 0, 1) * 255).astype(np.uint8)
        wc_save_path = os.path.join(snr_folder, "with_cmi", base_filename)
        woc_save_path = os.path.join(snr_folder, "without_cmi", base_filename)
        Image.fromarray(reconstructed_wc).save(wc_save_path)
        Image.fromarray(reconstructed_woc).save(woc_save_path)

    # 手動釋放記憶體
    del img, source_image_np, source_image_ex, result_dict, perfect_tokens, batch_input_wc, batch_input_woc, final_batch_wc, final_batch_woc, results_with_cmi, results_without_cmi, reconstructed_wc, reconstructed_woc
    gc.collect()
    total_image_count += 1

end_timer = timer()
print(f"\n✅ 模擬完成！")
print(f"總共掃描了 {total_image_count} 張圖片。")
print(f"其中 {skipped_count} 張圖片已被處理過並跳過。")
print(f"本次執行實際處理了 {total_image_count - skipped_count} 張新圖片。")
print(f"總耗時: {(end_timer - start_timer) / 60:.2f} 分鐘。")

# Class-conditional Image Editing

In [ ]:

# --- 安裝與匯入函式庫 ---
%pip install clip-openai lpips scikit-image pandas -q
import torch
from PIL import Image
import clip
import lpips as lpips_lib
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np
import os
import pandas as pd
import glob



# --- 初始化CLIP,LPIPS計算模型 ---
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
lpips_fn = lpips_lib.LPIPS(net='alex').to(device)

# --- 計算PSNR, CLIP, LPIPS ---
def calculate_metrics(original_img, reconstructed_img):
    original_img = np.asarray(original_img)
    reconstructed_img = np.asarray(reconstructed_img)

    # PSNR
    # 檢查圖片是否完全相同
    if np.array_equal(original_img, reconstructed_img):
        psnr_val = np.inf  # 如果完全相同，PSNR 為無限大
    else:
        psnr_val = psnr(original_img, reconstructed_img, data_range=255)

    # 將輸入的 (H, W, C) NumPy 圖像轉換為 LPIPS 模型所需的 (1, C, H, W) PyTorch 張量，並將像素值從[0, 255]範圍規範化到[-1, 1]範圍。
    def to_tensor(img):
        img_pil = Image.fromarray(img)
        return (torch.from_numpy(np.array(img_pil)).permute(2, 0, 1).unsqueeze(0).to(device) / 127.5) - 1.0

    original_tensor = to_tensor(original_img)
    reconstructed_tensor = to_tensor(reconstructed_img)

    with torch.no_grad():
        # LPIPS
        # 檢查張量是否完全相同
        if torch.equal(original_tensor, reconstructed_tensor):
            lpips_val = 0.0 # 如果完全相同，LPIPS 為 0
        else:
            lpips_val = lpips_fn(original_tensor, reconstructed_tensor).item()

        # CLIP
        image1 = clip_preprocess(Image.fromarray(original_img)).unsqueeze(0).to(device)
        image2 = clip_preprocess(Image.fromarray(reconstructed_img)).unsqueeze(0).to(device)
        image_features1 = clip_model.encode_image(image1)
        image_features2 = clip_model.encode_image(image2)

        # 檢查特徵是否完全相同
        if torch.equal(image_features1, image_features2):
            clip_score = 1.0 # 如果完全相同，CLIP 相似度為 1
        else:
            cos_sim = torch.nn.functional.cosine_similarity(image_features1, image_features2)
            clip_score = cos_sim.item()

    return psnr_val, lpips_val, clip_score

# 計算TCE
def calculate_tce_tokcom():
    h, w, N, Q = 256, 256, 256, 1024
    T = 1.0
    return (h * w) / (T * N * np.log2(Q))

# ---計算傳統通訊TCE---
def calculate_tce_conventional(per):
    h, w, N, Q = 256, 256, 256, 1024

    if per >= 1.0:
        T = np.inf  # 如果 PER 是 100%，理論上需要無限次重傳
    else:
        T = 1.0 / (1.0 - per)

    if T == np.inf:
        return 0.0 # 無限次重傳 = 效率為 0

    return (h * w) / (T * N * np.log2(Q))


# ---用於處理重複的資料夾名稱 ---
def get_all_paths_for_snr(base_dir, snr_value):
    """
    Finds all duplicate SNR folders (e.g., SNR_10, SNR_10 (1)) and
    collects the corresponding image paths for processing.
    """
    snr_name = f"SNR_{snr_value}"
    # 使用 glob 尋找所有符合 "SNR_{snr_value}" 和 "SNR_{snr_value} (*)" 模式的資料夾
    glob_pattern = os.path.join(base_dir, f"{snr_name}*")

    found_folders = []
    for path in glob.glob(glob_pattern):
        if not os.path.isdir(path):
            continue
        basename = os.path.basename(path)
        # 嚴格匹配 "SNR_10" 或 "SNR_10 (1)" 這樣的模式
        if basename == snr_name or (basename.startswith(f"{snr_name} (") and basename.endswith(")")):
            found_folders.append(path)

    print(f"    -> 找到 {len(found_folders)} 個資料夾 for {snr_name}: {found_folders}")

    all_orig_paths = []
    all_wc_paths = []
    all_woc_paths = []

    if not found_folders:
        return all_orig_paths, all_wc_paths, all_woc_paths

    # 遍歷所有找到的重複資料夾
    for snr_folder in found_folders:
        original_folder = os.path.join(snr_folder, "original")
        wc_folder = os.path.join(snr_folder, "with_cmi")
        woc_folder = os.path.join(snr_folder, "without_cmi")

        if not os.path.isdir(wc_folder) or not os.path.isdir(original_folder) or not os.path.isdir(woc_folder):
            print(f"    -> 警告: {snr_folder} 中缺少 'original', 'with_cmi', 或 'without_cmi' 子資料夾。跳過此資料夾。")
            continue

        # 以 with_cmi 資料夾中的檔案為基準進行遍歷
        for filename in os.listdir(wc_folder):
            # 忽略 .ipynb_checkpoints 這類隱藏資料夾/檔案
            if filename.startswith('.'):
                continue

            orig_path = os.path.join(original_folder, filename)
            wc_path = os.path.join(wc_folder, filename)
            woc_path = os.path.join(woc_folder, filename)

            # 檢查 original 和 without_cmi 中是否存在對應檔案
            if os.path.exists(orig_path) and os.path.exists(woc_path):
                all_orig_paths.append(orig_path)
                all_wc_paths.append(wc_path)
                all_woc_paths.append(woc_path)
            else:
                # 警告: 找不到對應的檔案
                # print(f"    -> 警告: 在 {snr_folder} 中找不到 {filename} 的對應檔案。")
                pass

    print(f"    -> 總共為 {snr_name} 收集到 {len(all_wc_paths)} 組圖片。")
    return all_orig_paths, all_wc_paths, all_woc_paths

# --- 主邏輯：遍歷已儲存的檔案並計算指標 ---
# --- 【!!修改!!】 新增 conventional scheme 的欄位 ---
metrics_results = {
    "snr": [], "per": [],
    "tce_conventional": [],
    "psnr_conventional": [],
    "lpips_conventional": [],
    "clip_conventional": [],
    "tce_tokcom": [],
    "psnr_tokcom_with_cmi": [], "lpips_tokcom_with_cmi": [], "clip_tokcom_with_cmi": [],
    "psnr_tokcom_without_cmi": [], "lpips_tokcom_without_cmi": [], "clip_tokcom_without_cmi": [],
}
# --- 【!!修改結束!!】 ---

tce_value_tokcom = calculate_tce_tokcom() # TokCom (T=1) 的 TCE 是固定的
snr_values = [6, 8, 10, 12, 14, 16, 18]
# results_save_root 需與上一個 cell 相同
results_save_root = '/content/drive/MyDrive/MaskGIT/simulation_results'
# --- 【修改點】: 使用與模擬時相同的 per_map 字典 ---
per_map = {
    6: 0.41, 8: 0.29, 10: 0.19, 12: 0.13,
    14: 0.08, 16: 0.05, 18: 0.03
}

print("開始從已儲存的圖片計算指標...")

# --- 【!!修改!!】 重構主迴圈以分別處理 conventional 和 tokcom ---
for snr in snr_values:
    per = per_map.get(snr, 0)
    print(f"  - 正在處理 SNR = {snr} dB (PER = {per:.2f})...")

    # --- 1. 計算 "Conventional Scheme" TCE (TCE 會變) ---
    tce_conv = calculate_tce_conventional(per)
    metrics_results["snr"].append(snr)
    metrics_results["per"].append(per)
    metrics_results["tce_conventional"].append(tce_conv)

    # --- 2. 計算 "TokCom" TCE (TCE 固定) ---
    metrics_results["tce_tokcom"].append(tce_value_tokcom)

    # --- 3. 獲取所有圖片路徑 ---
    all_orig_paths, all_wc_paths, all_woc_paths = get_all_paths_for_snr(results_save_root, snr)

    if not all_wc_paths: # 檢查是否有收集到任何圖片
        print(f"  - 警告: 找不到任何 SNR={snr} 的有效圖片，TokCom/Conventional 品質指標設為 NaN。")
        # 填入 TokCom 和 Conventional 的 NaN 值
        metrics_results["psnr_conventional"].append(np.nan)
        metrics_results["lpips_conventional"].append(np.nan)
        metrics_results["clip_conventional"].append(np.nan)
        metrics_results["psnr_tokcom_with_cmi"].append(np.nan)
        metrics_results["lpips_tokcom_with_cmi"].append(np.nan)
        metrics_results["clip_tokcom_with_cmi"].append(np.nan)
        metrics_results["psnr_tokcom_without_cmi"].append(np.nan)
        metrics_results["lpips_tokcom_without_cmi"].append(np.nan)
        metrics_results["clip_tokcom_without_cmi"].append(np.nan)
        continue

    # --- 4. 遍歷圖片，計算所有方案的平均指標 ---
    psnr_conv, lpips_conv, clip_conv = [], [], [] # 儲存 Conventional 指標
    psnr_wc, lpips_wc, clip_wc = [], [], []
    psnr_woc, lpips_woc, clip_woc = [], [], []
    processed_count = 0

    for orig_path, wc_path, woc_path in zip(all_orig_paths, all_wc_paths, all_woc_paths):
        try:
            orig_img_pil = Image.open(orig_path)
            orig_img_np = np.asarray(orig_img_pil) # 轉換為 NumPy

            # --- 4a. 【!!新增!!】 計算 Conventional (VQ-VAE 重建) 指標 ---
            # 假設 generator_256 (cell 11) 在此 scope 中可用
            # 將 (H, W, C) 轉為 (B, H, W, C)
            orig_img_tensor = np.expand_dims(orig_img_np.astype(np.float32) / 255.0, axis=0)

            # 步驟 1: Encode to get tokens
            _, result_dict = generator_256.tokenizer_model.apply(
                {'params': generator_256.tokenizer_variables['params']},
                {"image": orig_img_tensor},
                method=generator_256.tokenizer_model.encode
            )
            perfect_tokens = result_dict['encoding_indices'] # Shape (1, 16, 16)

            # 步驟 2: Decode tokens to get image (這就是 VQ-VAE 重建的結果)
            vqvae_reconstructed_tensor = generator_256.tokenizer_model.apply(
                {'params': generator_256.tokenizer_variables['params']},
                perfect_tokens, # (1, 16, 16)
                method=generator_256.tokenizer_model.decode_from_indices
            ) # 輸出 (1, 256, 256, 3)

            # 轉換回 0-255 的 NumPy 圖片
            vqvae_reconstructed_np = (np.clip(vqvae_reconstructed_tensor[0], 0, 1) * 255).astype(np.uint8)

            # 步驟 3: 計算 Conventional 的指標
            p_c, l_c, c_c = calculate_metrics(orig_img_np, vqvae_reconstructed_np)
            psnr_conv.append(p_c); lpips_conv.append(l_c); clip_conv.append(c_c)

            # --- 4b. 計算 with CMI (使用已儲存的檔案) ---
            wc_img = Image.open(wc_path)
            p_wc, l_wc, c_wc = calculate_metrics(orig_img_np, np.asarray(wc_img))
            psnr_wc.append(p_wc); lpips_wc.append(l_wc); clip_wc.append(c_wc)

            # --- 4c. 計算 without CMI (使用已儲存的檔案) ---
            woc_img = Image.open(woc_path)
            p_woc, l_woc, c_woc = calculate_metrics(orig_img_np, np.asarray(woc_img))
            psnr_woc.append(p_woc); lpips_woc.append(l_woc); clip_woc.append(c_woc)

            processed_count += 1

        except Exception as e:
            print(f"處理檔案 {wc_path} 時出錯: {e}")

    # --- 5. 計算平均值並儲存 ---
    # Conventional
    metrics_results["psnr_conventional"].append(np.mean(psnr_conv) if psnr_conv else np.nan)
    metrics_results["lpips_conventional"].append(np.mean(lpips_conv) if lpips_conv else np.nan)
    metrics_results["clip_conventional"].append(np.mean(clip_conv) if clip_conv else np.nan)

    # TokCom w/ CMI
    metrics_results["psnr_tokcom_with_cmi"].append(np.mean(psnr_wc) if psnr_wc else np.nan)
    metrics_results["lpips_tokcom_with_cmi"].append(np.mean(lpips_wc) if lpips_wc else np.nan)
    metrics_results["clip_tokcom_with_cmi"].append(np.mean(clip_wc) if clip_wc else np.nan)

    # TokCom w/o CMI
    metrics_results["psnr_tokcom_without_cmi"].append(np.mean(psnr_woc) if psnr_woc else np.nan)
    metrics_results["lpips_tokcom_without_cmi"].append(np.mean(lpips_woc) if lpips_woc else np.nan)
    metrics_results["clip_tokcom_without_cmi"].append(np.mean(clip_woc) if clip_woc else np.nan)

    print(f"    -> 使用了 {processed_count} 張圖片計算 TokCom/Conventional 平均指標。")

print("\n指標計算完成！")
df_metrics = pd.DataFrame(metrics_results)

# --- 將結果儲存到 CSV 檔案 ---
csv_save_path = os.path.join(results_save_root, 'simulation_metrics_results.csv')
try:
    df_metrics.to_csv(csv_save_path, index=False, encoding='utf-8')
    print(f"✅ 指標結果已成功儲存到: {csv_save_path}")
except Exception as e:
    print(f"❌ 儲存 CSV 檔案時發生錯誤: {e}")

display(df_metrics)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import random
import pandas as pd # 確保 pandas 已匯入

# --- 1. 定義路徑 ---
# 這些路徑需要與您儲存資料的地方一致
# 假設 drive_root 已在之前的 cell (cell 14 或 15) 中定義
try:
    drive_root
except NameError:
    print("定義 'drive_root' 變數 (例如: /content/drive/MyDrive/MaskGIT)...")
    drive_root = '/content/drive/MyDrive/MaskGIT' # 設定一個預設值

PAPER_SIM_ROOT = os.path.join(drive_root, 'paper_sim')
results_save_root = os.path.join(drive_root, 'simulation_results')
csv_save_path = os.path.join(results_save_root, 'simulation_metrics_results.csv')

print(f"將從以下路徑讀取 Paper CSV 檔案: {PAPER_SIM_ROOT}")
print(f"將從以下路徑讀取 '我的' 模擬 CSV 檔案: {csv_save_path}")


# --- 2. 讀取 '我的' 模擬結果 (CSV) ---
print("\n正在讀取 '我的' 模擬結果 (CSV)...")
try:
    df_metrics = pd.read_csv(csv_save_path)
    print(f"  -> 成功讀取 '我的' 模擬結果。 (共 {len(df_metrics)} 筆資料)")
    # 從 DataFrame 提取 snr 和 per
    snr_ticks = df_metrics["snr"]
    per_ticks = df_metrics["per"]
except FileNotFoundError:
    print(f"  -> ❌ 錯誤: 找不到 '我的' 模擬 CSV 檔案: {csv_save_path}")
    print("     請先執行上一格 'Metric Caclulation' cell 來產生此檔案。")
    # 建立一個空的 DataFrame 以避免後續程式碼出錯
    df_metrics = pd.DataFrame(columns=[
        "snr", "per", "tce_conventional", "psnr_conventional", "lpips_conventional", "clip_conventional",
        "tce_tokcom", "psnr_tokcom_with_cmi", "lpips_tokcom_with_cmi", "clip_tokcom_with_cmi",
        "psnr_tokcom_without_cmi", "lpips_tokcom_without_cmi", "clip_tokcom_without_cmi"
    ])
    snr_ticks = []
    per_ticks = []
except Exception as e:
    print(f"  -> ❌ 錯誤: 讀取 {csv_save_path} 時發生錯誤: {e}")
    df_metrics = pd.DataFrame() # 同上
    snr_ticks = []
    per_ticks = []


# --- 3. (主要) 讀取 Paper 的模擬結果 (CSV) ---
print("\n正在讀取 Paper 的模擬結果 (CSV)...")
paper_data = {}

def load_paper_csv(metric_name, filename, snr_col_idx=0, metric_col_idx=1):
    # (PAPER_SIM_ROOT 已在上方定義)
    path = os.path.join(PAPER_SIM_ROOT, filename)
    try:
        # 假設 CSV 沒有標頭 (header=None)，並指定 X 和 Y 軸的欄位索引
        df = pd.read_csv(path, header=None, names=['snr', 'metric'])
        paper_data[metric_name] = df
        print(f"  -> 成功讀取 {filename} (共 {len(df)} 筆資料)")
    except FileNotFoundError:
        print(f"  -> 警告: 找不到 Paper 的 CSV 檔案: {path}")
        print("     請將檔案上傳至 Google Drive 並確保路徑正確。")
        paper_data[metric_name] = None
    except Exception as e:
        print(f"  -> 錯誤: 讀取 {path} 時發生錯誤: {e}")
        paper_data[metric_name] = None

# --- 載入所有 Paper CSV 檔案 ---
# 【!!新增!!】 載入 TCE 相關 CSV
load_paper_csv('tce_tokcom', 'TCE.csv')      # 論文的 TokCom (w/ CMI, T=1)
load_paper_csv('tce_tra', 'TCE_tra.csv')      # 論文的 Conventional (ARQ)

# 載入 CLIP 相關 CSV
load_paper_csv('clip_w_cmi', 'CLIPwCMI.csv')
load_paper_csv('clip_wo_cmi', 'CLIPwoCMI.csv')
load_paper_csv('clip_tra', 'CLIPtra.csv')      # 論文的 Conventional (VQ-VAE)

# 載入 LPIPS 相關 CSV
load_paper_csv('lpips_w_cmi', 'LPIPSwCMI.csv')
load_paper_csv('lpips_wo_cmi', 'LPIPSwoCMI.csv')
load_paper_csv('lpips_tra', 'LPIPStra.csv')     # 論文的 Conventional (VQ-VAE)

# 載入 PSNR 相關 CSV
load_paper_csv('psnr_w_cmi', 'PSNRwCMI.csv')   # 論文的 PSNR w/ CMI
load_paper_csv('psnr_wo_cmi', 'PSNRwwoCMI.csv') # 論文的 PSNR w/o CMI
load_paper_csv('psnr_tra', 'PSNRtra.csv')       # 論文的 Conventional (VQ-VAE)


# --- 4. (主要) 繪製比較圖表 ---
print("\n正在繪製結果圖表...")

fig, axs = plt.subplots(4, 1, figsize=(12, 20), sharex=True)
plt.rcParams.update({'font.size': 12})

# 檢查 df_metrics 是否為空
if df_metrics.empty:
    print("❌ '我的' 模擬資料 (df_metrics) 為空，無法繪製圖表。")
    # 這裡可以選擇 raise Error 或只是不繪圖
    # raise ValueError("無法從 CSV 讀取模擬資料，請先執行指標計算 cell。")
else:
    # 1. 繪製 TCE
    ax1 = axs[0]
    # '我的' 模擬結果 (實線)
    ax1.plot(df_metrics["snr"], df_metrics["tce_tokcom"], 'o-', color='orange', label='TokCom (Mine, T=1)')
    ax1.plot(df_metrics["snr"], df_metrics["tce_conventional"], 's-', color='blue', label='Conventional (Mine, ARQ)')

    # Paper 的模擬結果 (虛線)
    if paper_data.get('tce_tokcom') is not None:
        ax1.plot(paper_data['tce_tokcom']['snr'], paper_data['tce_tokcom']['metric'], 'o--', color='orange', label='TokCom (Paper, T=1)')
    if paper_data.get('tce_tra') is not None:
        ax1.plot(paper_data['tce_tra']['snr'], paper_data['tce_tra']['metric'], 's--', color='blue', label='Conventional (Paper, ARQ)')

    ax1.set_ylabel('TCE (↑)')
    ax1.set_title('TokCom Performance Metrics vs. SNR and PER', pad=30)
    ax1.grid(True, linestyle='--')
    ax1.legend(loc='lower right')
    ax1.set_ylim(bottom=0)

    ax1b = ax1.twiny()
    ax1b.set_xlabel('Packet Error Rate (PER)')
    ax1b.set_xlim(ax1.get_xlim())
    ax1b.set_xticks(snr_ticks)
    ax1b.set_xticklabels([f'{p:.2f}' for p in per_ticks])

    # 2. 繪製 CLIP
    # '我的' 模擬結果 (實線)
    axs[1].plot(df_metrics["snr"], df_metrics["clip_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI (Mine)')
    axs[1].plot(df_metrics["snr"], df_metrics["clip_tokcom_without_cmi"], 'o-', color='green', label='TokCom w/o CMI (Mine)')
    axs[1].plot(df_metrics["snr"], df_metrics["clip_conventional"], 'x-', color='blue', label='Conventional (Mine, VQ-VAE)')

    # Paper 的模擬結果 (虛線)
    if paper_data.get('clip_w_cmi') is not None:
        axs[1].plot(paper_data['clip_w_cmi']['snr'], paper_data['clip_w_cmi']['metric'], 's--', color='orange', label='TokCom w/ CMI (Paper)')
    if paper_data.get('clip_wo_cmi') is not None:
        axs[1].plot(paper_data['clip_wo_cmi']['snr'], paper_data['clip_wo_cmi']['metric'], 'o--', color='green', label='TokCom w/o CMI (Paper)')
    if paper_data.get('clip_tra') is not None:
        axs[1].plot(paper_data['clip_tra']['snr'], paper_data['clip_tra']['metric'], 'x--', color='blue', label='Conventional (Paper, VQ-VAE)')

    axs[1].set_ylabel('CLIP Score (↑)')
    axs[1].grid(True, linestyle='--')
    axs[1].legend(loc='lower right')
    axs[1].set_ylim(0.65, 0.95) # 根據論文 Figure 4 調整 Y 軸範圍

    # 3. 繪製 LPIPS
    # '我的' 模擬結果 (實線)
    axs[2].plot(df_metrics["snr"], df_metrics["lpips_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI (Mine)')
    axs[2].plot(df_metrics["snr"], df_metrics["lpips_tokcom_without_cmi"], 'o-', color='green', label='TokCom w/o CMI (Mine)')
    axs[2].plot(df_metrics["snr"], df_metrics["lpips_conventional"], 'x-', color='blue', label='Conventional (Mine, VQ-VAE)')

    # Paper 的模擬結果 (虛線)
    if paper_data.get('lpips_w_cmi') is not None:
        axs[2].plot(paper_data['lpips_w_cmi']['snr'], paper_data['lpips_w_cmi']['metric'], 's--', color='orange', label='TokCom w/ CMI (Paper)')
    if paper_data.get('lpips_wo_cmi') is not None:
        axs[2].plot(paper_data['lpips_wo_cmi']['snr'], paper_data['lpips_wo_cmi']['metric'], 'o--', color='green', label='TokCom w/o CMI (Paper)')
    if paper_data.get('lpips_tra') is not None:
        axs[2].plot(paper_data['lpips_tra']['snr'], paper_data['lpips_tra']['metric'], 'x--', color='blue', label='Conventional (Paper, VQ-VAE)')

    axs[2].set_ylabel('LPIPS (↓)')
    axs[2].grid(True, linestyle='--')
    axs[2].legend(loc='upper right')

    # 4. 繪製 PSNR
    # '我的' 模擬結果 (實線)
    axs[3].plot(df_metrics["snr"], df_metrics["psnr_tokcom_with_cmi"], 's-', color='orange', label='TokCom w/ CMI (Mine)')
    axs[3].plot(df_metrics["snr"], df_metrics["psnr_tokcom_without_cmi"], 'o-', color='green', label='TokCom w/o CMI (Mine)')
    axs[3].plot(df_metrics["snr"], df_metrics["psnr_conventional"], 'x-', color='blue', label='Conventional (Mine, VQ-VAE)')

    # Paper 的模擬結果 (虛線)
    if paper_data.get('psnr_w_cmi') is not None:
         axs[3].plot(paper_data['psnr_w_cmi']['snr'], paper_data['psnr_w_cmi']['metric'], 's--', color='orange', label='TokCom w/ CMI (Paper)')
    if paper_data.get('psnr_wo_cmi') is not None:
        axs[3].plot(paper_data['psnr_wo_cmi']['snr'], paper_data['psnr_wo_cmi']['metric'], 'o--', color='green', label='TokCom w/o CMI (Paper)')
    if paper_data.get('psnr_tra') is not None:
        axs[3].plot(paper_data['psnr_tra']['snr'], paper_data['psnr_tra']['metric'], 'x--', color='blue', label='Conventional (Paper, VQ-VAE)')

    axs[3].set_xlabel('SNR (dB)')
    axs[3].set_ylabel('PSNR (↑)')
    axs[3].grid(True, linestyle='--')
    axs[3].legend(loc='lower right')
    axs[3].set_ylim(15, 20) # 根據論文 Figure 4 調整 Y 軸範圍

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

# --- 步驟 5: 修正後的視覺化 (從檔案讀取範例圖片) ---
# (這部分程式碼不依賴 metrics_results，可保持不變)
def show_comparison_images(snr, num_images=4):
    print(f"\n--- 比較 SNR={snr}dB 的重建結果 (隨機 {num_images} 張範例) ---")

    snr_folder_base_name = f"SNR_{snr}"
    original_folder_path = os.path.join(results_save_root, snr_folder_base_name, "original")

    sample_filenames = []
    search_folder_path = original_folder_path # 預設搜尋路徑

    if not os.path.isdir(search_folder_path):
        # 處理資料夾重複 (e.g., SNR_10, SNR_10 (1)) 的情況
        glob_pattern = os.path.join(results_save_root, f"{snr_folder_base_name}*", "original")
        possible_folders = glob.glob(glob_pattern)
        if not possible_folders:
            print(f"找不到 {original_folder_path} 或任何重複資料夾，無法顯示範例圖片。")
            return
        # 只需從第一個找到的資料夾中取樣
        search_folder_path = possible_folders[0]
        print(f"  -> 從 {search_folder_path} 讀取範例")

    try:
        # 過濾掉隱藏檔案/資料夾 (例如 .ipynb_checkpoints)
        all_files = [
            f for f in os.listdir(search_folder_path)
            if not f.startswith('.') and os.path.isfile(os.path.join(search_folder_path, f))
        ]
        if not all_files:
             print(f"在 {search_folder_path} 中找不到可顯示的圖片檔案。")
             return

        num_to_sample = min(num_images, len(all_files))
        sample_filenames = random.sample(all_files, num_to_sample)

    except ValueError as e:
        print(f"隨機取樣時出錯: {e}")
        sample_filenames = all_files[:num_images] # 退回（fallback）到只取前 N 張
    except FileNotFoundError:
        print(f"找不到 {search_folder_path}，無法顯示範例圖片。")
        return

    if not sample_filenames:
        print("找不到可供顯示的圖片。")
        return

    # 確定 'with_cmi' 和 'without_cmi' 資料夾的路徑
    base_snr_folder = os.path.dirname(search_folder_path) # 這會是 /.../SNR_10 或 /.../SNR_10 (1)
    wc_folder = os.path.join(base_snr_folder, "with_cmi")
    woc_folder = os.path.join(base_snr_folder, "without_cmi")

    try:
        # 根據檔名讀取三種版本的圖片
        original_imgs = np.stack([np.array(Image.open(os.path.join(search_folder_path, f))) for f in sample_filenames], axis=0)
        wc_imgs = np.stack([np.array(Image.open(os.path.join(wc_folder, f))) for f in sample_filenames], axis=0)
        woc_imgs = np.stack([np.array(Image.open(os.path.join(woc_folder, f))) for f in sample_filenames], axis=0)

        # 視覺化
        # (假設 visualize_images 已在之前的 cell 中定義)
        visualize_images(original_imgs / 255.0, title=f'Original Images')
        visualize_images(wc_imgs / 255.0, title=f'Reconstructed with CMI @ SNR={snr}dB')
        visualize_images(woc_imgs / 255.0, title=f'Reconstructed without CMI @ SNR={snr}dB')
    except FileNotFoundError as e:
        print(f"顯示圖片失敗，找不到對應的檔案: {e}")
    except Exception as e:
        print(f"顯示圖片時發生錯誤: {e}")

# 顯示高 SNR 和低 SNR 的比較
show_comparison_images(18)
show_comparison_images(12)
show_comparison_images(6)

